# 🚀 Тестирование Agent Framework

Этот ноутбук пошагово проверяет все компоненты фреймворка агентов:

1. **Тест базовых функций** - проверка инструментов
2. **Test Agent** - простейший агент (1 состояние)
3. **My Agent** - переходы между состояниями (2 состояния)
4. **Router Agent** - условный роутинг (ветвление)
5. **Multi-Agent System** - один агент вызывает другого
6. **Audit Agent** - полный тест (6 состояний)

Бэкенд (GigaChat или LM Studio) настраивается в `config.yaml`.

In [1]:
# # Установка зависимостей (запустите один раз)
# %uv pip install -r requirements.txt

## 📦 Общая настройка

Импорты и конфигурация для всех тестов.

In [1]:
# Импорты
import yaml

# Импорты агентов
from src.agents.test_agent import TestAgent
from src.agents.my_agent import MyAgent
from src.agents.router_agent import RouterAgent
from src.agents.supervisor_agent import SupervisorAgent
from src.agents.audit_agent import AuditAgent

# Импорты инструментов и клиентов
from src.tools.tools import (
    get_tools_dict,
    reset_memory,
    register_agent,
    calculator,
)
from src.connections.clients import get_llm_client

# Logging (настройка уровня в config.yaml -> logging.level)
from src.agent_engine import init_logging

print("✓ Импорты выполнены успешно")

# Загрузка конфигурации
with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

backend = config['active_backend']
recursion_limit = config['agent']['recursion_limit']

print(f"Активный бэкенд: {backend}")
print(f"Лимит рекурсии графа: {recursion_limit}")

# Создание LLM клиента
llm = get_llm_client(backend, config)
print(f"✓ LLM клиент создан: {type(llm).__name__}")

# Инициализация logging (уровень задаётся в config.yaml)
init_logging()
print("✓ Logging инициализирован")

✓ Импорты выполнены успешно
Активный бэкенд: lmstudio
Лимит рекурсии графа: 30
✓ LLM клиент создан: ChatOpenAI
logging: level=detailed
✓ Logging инициализирован


---

## 1️⃣ Тест базовых функций

Проверяем что инструменты работают корректно.

In [2]:
print("=" * 60)
print("ТЕСТ 1: Базовые функции")
print("=" * 60)

# Тест калькулятора
print("\n1. Тест калькулятора:")
result = calculator.invoke("2 ** 10")
print(f"   2^10 = {result}")
assert result == "1024", "Калькулятор работает неправильно!"
print("   ✓ Калькулятор работает")

# Тест памяти
print("\n2. Тест памяти:")
from src.tools.tools import memory
reset_memory()
memory.invoke({"action": "save", "key": "test_key", "value": "test_value"})
result = memory.invoke({"action": "get", "key": "test_key"})
assert result.get("ok") and result.get("entries", {}).get("test_key") == "test_value", (
    "Память работает неправильно!"
)
print("   ✓ Память работает")

print("\n✅ Все базовые функции работают корректно!")

ТЕСТ 1: Базовые функции

1. Тест калькулятора:
   2^10 = 1024
   ✓ Калькулятор работает

2. Тест памяти:
   ✓ Память работает

✅ Все базовые функции работают корректно!


In [5]:
# print("=" * 60)
# print("ТЕСТ 1.5: Проверка подключения к LLM")
# print("=" * 60)

# from langchain_core.messages import HumanMessage

# try:
#     test_response = llm.invoke([HumanMessage(content="Ответь одним словом: 2+2=")])
#     print(f"\n✓ LLM ответил: {test_response.content.strip()}")
#     print("✓ Подключение к LLM работает")
# except Exception as e:
#     print(f"\n✗ Ошибка подключения к LLM: {e}")
#     print("Проверьте что LM Studio запущен и доступен на порту из config.yaml")

---

## 2️⃣ Test Agent - Простейший агент

Граф: `[work] → END`

Проверяем:
- Создание агента через AgentConfig
- Работу с инструментами
- Переход в END по ключевому слову

In [4]:
print("======== ТЕСТ 2: Test Agent (1 состояние) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="test_agent")

# Создание агента
test_agent = TestAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {test_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(test_agent.visualize())

======== ТЕСТ 2: Test Agent (1 состояние) ========

✓ Агент создан: TestAgent(id=TestAgent_2362420178352, states=1)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 1
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Единственное рабочее состояние
    Переходы: END
    Инструменты (shared): calculator, memory, think
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [5]:
result = test_agent.invoke([
    "Посчитай 15 * 23 и сохрани результат в память с ключом 'result'"
])

RUN 1a251403 | TestAgent_2362420178352 | started
STATE START -> work
  REQ 1 | msgs=2 in=701 out=45
    [SYS] Ты простой тестовый агент.

Твои возможности:
- calculator: вычисление математических выражений
- memory: сохранение/чтение данных
- think: размышления

Выполни запрос пользователя, используя доступные инструменты.


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему переходишь именно туда.
- summary: что было сделано и что нужно сделать в следующем состоянии. Укажи какие ключи сохранены в memory.
- next_state: одно из доступных значений:
  - "stay" — остаться в текущем состоянии (нужно ещё поработать)
  - "END"

ВАЖНО: после вызова transition не вызывай другие инструменты — напиши финальный ответ.
    [USER] Посчитай 15 * 23 и сохрани результат в память с ключом 'result'
  TOOL calculator params={'expression': '15 * 23'}
  TOOL memory params={'action': 'save', 'key': 'calculator_last', 'value': ' 15 * 23 

---

## 3️⃣ My Agent - Переходы между состояниями

Граф: `[work] ⟳ → [summarize] → END`

Проверяем:
- Циклическое состояние (work возвращается в себя)
- Условный переход по ключевому слову
- Работу с памятью между состояниями

In [5]:
print("======== ТЕСТ 3: My Agent (2 состояния + переход) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="my_agent")

# Создание агента
my_agent = MyAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {my_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(my_agent.visualize())

======== ТЕСТ 3: My Agent (2 состояния + переход) ========

✓ Агент создан: MyAgent(id=MyAgent_1484570122704, states=2)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 2
  - Уникальных тулов: 5
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Основное рабочее состояние с полным набором инструментов
    Переходы: summarize
    Инструменты (shared): calculator, ask_human, think, memory
    Memory injections: нет
  - summarize
    Описание: Финальное состояние для рефлексии и подведения итогов
    Переходы: END
    Инструменты (shared): summarize, memory
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [ ]:
# Запуск агента
result = my_agent.invoke([
    "давай подсчитаем площадь круга?"
])

RUN 80e5acd6 | MyAgent_1484570122704 | started
STATE START -> work
  REQ 1 | msgs=2 in=829 out=37
    [SYS] Ты полезный помощник для математических вычислений.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Сохрани важные результаты через memory


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему переходишь именно туда.
- summary: что было сделано и что нужно сделать в следующем состоянии. Укажи какие ключи сохранены в memory.
- next_state: одно из доступных значений:
  - "stay" — остаться в текущем состоянии 

---

## 4️⃣ Router Agent - Условный роутинг

Граф: `[classify] → [math | text | error] → END`

Проверяем:
- Классификацию запроса
- Роутер с несколькими выходами
- Разные пути обработки

In [ ]:
print("======== ТЕСТ 4: Router Agent (условный роутинг) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="router_agent")

# Создание агента
router_agent = RouterAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {router_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(router_agent.visualize())

ТЕСТ 4: Router Agent (условный роутинг)

✓ Агент создан: RouterAgent(id=RouterAgent_2143587284992, states=4)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: classify
  - Состояний: 4
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - classify (entry)
    Описание: Классификация типа запроса
    Переходы: math, text, error
    Инструменты: think [shared], memory [shared]
    Memory injections: request_type
  - math
    Описание: Обработка математических запросов
    Переходы: END
    Инструменты: calculator [shared], memory [shared], think [shared]
    Memory injections: request_type, result
  - text
    Описание: Обработка текстовых запросов
    Переходы: END
    Инструменты: memory [shared], think [shared]
    Memory injections: request_type, response_text
  - error
    Описание: Обработка нераспознанных запросов
    Переходы: END
    Инструменты: think [shared]
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найден

In [ ]:
# Тест 1: Математический запрос
print("======== Тест 4.1: Математический запрос ========")

reset_memory()
result = router_agent.invoke(["Посчитай 5 + 5"])

In [ ]:
# Тест 2: Текстовый запрос
print("======== Тест 4.2: Текстовый запрос ========")

reset_memory()
result = router_agent.invoke(["Привет! Как дела?"])

---

## 5️⃣ Multi-Agent System - Композиция агентов

Граф: `Supervisor [delegate] → [aggregate] → END`
         ↓ вызывает ↓
       Test Agent, Router Agent

Проверяем:
- Регистрацию агентов в реестре
- Вызов агента из агента через инструмент call_agent
- Агрегацию результатов от нескольких агентов

In [ ]:
print("======== ТЕСТ 5: Multi-Agent System (композиция агентов) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="supervisor_agent")

# Создаем подчиненных агентов
test_agent_sub = TestAgent(llm, tools_dict, agent_id="test_agent_sub")
router_agent_sub = RouterAgent(llm, tools_dict, agent_id="router_agent_sub")

print("\n✓ Подчиненные агенты созданы:")
print(f"  - {test_agent_sub}")
print(f"  - {router_agent_sub}")

# Регистрируем их в реестре
register_agent("test_agent", test_agent_sub)
register_agent("router_agent", router_agent_sub)

print("\n✓ Агенты зарегистрированы в реестре")

ТЕСТ 5: Multi-Agent System (композиция агентов)

✓ Подчиненные агенты созданы:
  - TestAgent(id=test_agent_sub, states=1)
  - RouterAgent(id=router_agent_sub, states=4)

✓ Агенты зарегистрированы в реестре


In [4]:
# Создаем супервизора с инструментом call_agent
supervisor_tools_dict = get_tools_dict(agent_name="supervisor_agent")
supervisor = SupervisorAgent(llm, supervisor_tools_dict)

print(f"✓ Супервизор создан: {supervisor}")

# Визуализация графа
print("\nСтруктура графа супервизора:")
print(supervisor.visualize())

✓ Супервизор создан: SupervisorAgent(id=SupervisorAgent_2144945434272, states=2)

Структура графа супервизора:
Граф агента (preflight):

Сводка:
  - Entry point: delegate
  - Состояний: 2
  - Уникальных тулов: 4
  - Ключей в памяти сейчас: 0

Состояния:
  - delegate (entry)
    Описание: Делегирование задач специализированным агентам
    Переходы: aggregate
    Инструменты: call_agent [shared], memory [shared], think [shared]
    Memory injections: request_type, response_text, delegation_notes
  - aggregate
    Описание: Агрегация результатов от агентов
    Переходы: END
    Инструменты: memory [shared], summarize [shared], think [shared]
    Memory injections: request_type, response_text, delegation_notes

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [ ]:
# Запуск супервизора
print("\n" + "=" * 60)
print("Запуск: Посчитай 10 * 5 и поздоровайся")
print("=" * 60 + "\n")

result = supervisor.invoke([
    "Посчитай 10 * 5 через test_agent и передай приветствие через router_agent"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Multi-Agent System работает корректно!")

---

## 6️⃣ Audit Agent - Полный тест

Граф: `[start_work] → [analize_word] → [analize_sql] → [analize_py] → [self_check] ⟳ → [write_report] → END`

Проверяем:
- Сложный многошаговый workflow
- Роутер с самопроверкой
- Работу с файловой системой
- Анализ разных типов файлов
- Пакетный **memory** в `start_work`: после `list_case_files` шесть ключей (`case_id` … `py_files`) сохраняются **одним** вызовом `memory(action="save", keys=[...], values=[...])` (см. промпт `AuditAgent` и запрос в ячейке ниже)

In [2]:
print("======== ТЕСТ 6: Audit Agent (полный тест) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="audit_agent")

# Создание агента
audit_agent = AuditAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {audit_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(audit_agent.visualize())

======== ТЕСТ 6: Audit Agent (полный тест) ========

✓ Агент создан: AuditAgent(id=AuditAgent_2055487539440, states=5)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: start_work
  - Состояний: 5
  - Уникальных тулов: 9
  - Ключей в памяти сейчас: 0

Состояния:
  - start_work (entry)
    Описание: Поиск папки проверки и сбор списка файлов
    Переходы: analize_sql
    Инструменты (shared): ask_human, think, memory
    Инструменты (local): list_data_folders, find_case_folder, list_case_files
    Memory injections: case_id, case_folder, case_files, docx_files, sql_files, py_files
  - analize_sql
    Описание: Анализ SQL скриптов на риски и проблемы
    Переходы: analize_py, start_work
    Инструменты (shared): memory, think
    Инструменты (local): read_sql_file
    Memory injections: case_files, sql_files, sql_verdicts
  - analize_py
    Описание: Анализ Python скриптов на риски и проблемы
    Переходы: self_check
    Инструменты (shared): memory, think
    Инструмен

In [3]:
print("\nВнимание: Агент будет задавать вопросы!")
print("Попробуйте ввести: 99-41116 или 99-41158\n")

result = audit_agent.invoke([
    "Проведи аудит файлов проверки. В состоянии start_work после получения списка файлов "
    "сохрани case_id, case_folder, case_files, docx_files, sql_files и py_files "
    "одним вызовом memory с параметрами keys и values (одинаковая длина списков)."
])

# Вывод результата


Внимание: Агент будет задавать вопросы!
Попробуйте ввести: 99-41116 или 99-41158

RUN 0fac4241 | AuditAgent_2055487539440 | started
STATE START -> start_work
  REQ 1 | msgs=2 in=1308 out=42
    [SYS] Ты агент аудита файлов проверки.
Твоя цель на этом шаге: найти папку проверки и собрать список файлов.

Алгоритм:
1. Попроси у пользователя номер проверки, если у тебя его нет.
2. Используй инструмент list_data_folders, чтобы увидеть доступные папки.
3. Используй find_case_folder, чтобы сопоставить ввод пользователя с папками.
4. Если статус "needs_confirmation" или есть сомнения — задай уточняющий вопрос.
5. Когда папка подтверждена, используй list_case_files и сохрани в память ВСЕ шесть ключей
   ОДНИМ вызовом memory (пакетно), без нескольких подряд вызовов memory(save) на отдельные ключи:
   memory(
     action="save",
     keys=["case_id", "case_folder", "case_files", "docx_files", "sql_files", "py_files"],
     values=[<номер проверки>, <путь к папке кейса>, <полный список файлов>, <

KeyboardInterrupt: 

---

## 🎉 Итоги тестирования

Если все тесты пройдены, значит фреймворк агентов работает корректно!

### Что было протестировано:

✅ **Базовые функции** - инструменты работают  
✅ **Test Agent** - простейший агент с 1 состоянием  
✅ **My Agent** - переходы между состояниями  
✅ **Router Agent** - условный роутинг  
✅ **Multi-Agent** - композиция агентов  
✅ **Audit Agent** - сложный workflow  

### Архитектура:

- **AgentConfig** - базовый класс для всех агентов
- **State** - декларативное описание состояний
- **Transition** - описание переходов
- **Изолированная история** - каждый агент имеет свою историю
- **Глобальная память** - инструменты memory доступны всем
- **Реестр агентов** - для мультиагентных систем

### Следующие шаги:

1. Создавайте новых агентов копированием папки и редактированием `agent.py`
2. Используйте композицию через `register_agent()` и `call_agent()`
3. Настраивайте логирование в `config.yaml` → `logging.level`: `off` / `simple` / `detailed`

In [ ]:
print("=" * 60)
print("🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!")
print("=" * 60)
print("\nФреймворк агентов готов к использованию!")
print("\nДокументация: README.md")
print("Примеры агентов: agents/*/agent.py")

🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!

Фреймворк агентов готов к использованию!

Документация: README.md
Примеры агентов: agents/*/agent.py


## Streamlit UI

Графический интерфейс для запуска агентов и общения с ними через браузер.  
Файл: `src/ui/streamlit_ui.py`

**Возможности:**
- Выбор агента из списка и запуск с произвольным сообщением
- Лента событий в реальном времени (вызовы инструментов, ответы LLM, переходы состояний)
- Вопрос/ответ — вопросы агента появляются в интерфейсе, ответ вводится там же
- Кнопка Стоп + Перезапуск — остановить агента и запустить заново в любой момент

In [2]:
import subprocess, sys, time, socket
from pathlib import Path

_PID_FILE = Path(".streamlit.pid")
_PORT = 8501

def _port_open(port: int) -> bool:
    try:
        with socket.create_connection(("localhost", port), timeout=1):
            return True
    except OSError:
        return False

# Если Streamlit уже запущен — не запускаем второй экземпляр
if _port_open(_PORT):
    print(f"✅ Streamlit уже работает: http://localhost:{_PORT}")
    print("   Чтобы перезапустить — сначала выполните ячейку ОСТАНОВКИ ниже, затем эту снова.")
else:
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "src/ui/streamlit_ui.py",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    _PID_FILE.write_text(str(proc.pid))
    print(f"Запускаем Streamlit (PID {proc.pid}), ждём готовности", end="", flush=True)

    for _ in range(90):          # ждём до 90 секунд (тяжёлые импорты)
        time.sleep(1)
        print(".", end="", flush=True)
        if proc.poll() is not None:  # процесс упал
            print(f"\n❌ Streamlit завершился с кодом {proc.returncode}.")
            print("Запустите в терминале: streamlit run src/ui/streamlit_ui.py  — чтобы увидеть ошибку.")
            break
        if _port_open(_PORT):
            print(f"\n✅ Готово! Откройте: http://localhost:{_PORT}")
            break
    else:
        print(f"\n⚠️ Превышено время ожидания. Попробуйте вручную: http://localhost:{_PORT}")


Запускаем Streamlit (PID 28172), ждём готовности.
✅ Готово! Откройте: http://localhost:8501


In [3]:
import os, sys, signal, subprocess, socket, time
from pathlib import Path

_PID_FILE = Path(".streamlit.pid")
_PORT = 8501

def _port_open(port: int) -> bool:
    try:
        with socket.create_connection(("localhost", port), timeout=1):
            return True
    except OSError:
        return False

def _kill_pid(pid: int) -> bool:
    """Завершает процесс по PID. Возвращает True если успешно."""
    try:
        if sys.platform == "win32":
            r = subprocess.run(["taskkill", "/PID", str(pid), "/F", "/T"],
                               capture_output=True)
            return r.returncode == 0
        else:
            os.kill(pid, signal.SIGTERM)
            return True
    except (subprocess.CalledProcessError, ProcessLookupError, OSError):
        return False

def _find_pid_by_port(port: int) -> int | None:
    """Ищет PID процесса слушающего порт. Кроссплатформенно."""
    if sys.platform == "win32":
        result = subprocess.run(
            ["powershell", "-ExecutionPolicy", "Bypass", "-Command",
             f"(Get-NetTCPConnection -LocalPort {port} -ErrorAction SilentlyContinue).OwningProcess"],
            capture_output=True, text=True
        )
    else:
        result = subprocess.run(["lsof", "-ti", f"tcp:{port}"],
                                capture_output=True, text=True)
    pid_str = result.stdout.strip().splitlines()[0] if result.stdout.strip() else ""
    return int(pid_str) if pid_str.isdigit() else None

# --- Остановка ---
# Шаг 1: убиваем по PID-файлу (если есть); файл удаляем всегда (в т.ч. при битом содержимом)
if _PID_FILE.exists():
    try:
        pid = int(_PID_FILE.read_text().strip())
        _kill_pid(pid)
    except ValueError:
        pass
    finally:
        _PID_FILE.unlink(missing_ok=True)

# Шаг 2: всегда проверяем порт — убиваем что осталось
time.sleep(0.5)
if _port_open(_PORT):
    pid = _find_pid_by_port(_PORT)
    if pid:
        if _kill_pid(pid):
            print(f"Streamlit (PID {pid}) остановлен.")
        else:
            print(f"Не удалось остановить процесс {pid}. Попробуйте вручную.")
    else:
        print(f"Порт {_PORT} занят неизвестным процессом.")
else:
    print(f"Streamlit остановлен — порт {_PORT} свободен.")

Streamlit остановлен — порт 8501 свободен.
